In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

print("GROQ_API_KEY is set:", os.environ.get("GROQ_API_KEY") is not None)

GROQ_API_KEY is set: True


Preparation

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = [file.parse() for file in files]

Q1

In [5]:
print(f"Number of documents: {len(documents)}")

Number of documents: 72


Q2

In [8]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(query, num_results=3)
print(f"Filename of first search result: {search_results[0]['filename']}")

Filename of first search result: 01-agentic-rag/lessons/14-agentic-loop.md


Q3 RAG

In [14]:
from groq import Groq

# Initialize the Groq client
client = Groq()

context = ""
for doc in search_results:
    context += f"Filename: {doc['filename']}\nContent: {doc['content']}\n\n"

prompt = f"""
You are a course assistant. Answer the question based on the context provided.

Context:
{context}

Question: {query}
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(f"Prompt input: {response.usage.prompt_tokens}.")

Prompt input: 5772.


Q4 Chunking

In [16]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


Q5 RAG with Chunking

In [19]:
index_chunks = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)

search_results_chunks = index_chunks.search(query, num_results=3)

context_chunks = ""
for doc in search_results_chunks:
    context_chunks += f"Filename: {doc['filename']}\nContent: {doc['content']}\n\n"

prompt_chunks = f"""
You are a course assistant. Answer the question based on the context provided.
Context:
{context_chunks}
Question: {query}
"""
response_chunks = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt_chunks}
    ]
)

difference = response.usage.prompt_tokens / response_chunks.usage.prompt_tokens
print(f"Prompt input for chunks: {response_chunks.usage.prompt_tokens}.")
print(f"Difference: {difference:.1f}x fewer tokens used with chunking.")


Prompt input for chunks: 1501.
Difference: 3.8x fewer tokens used with chunking.


Q5 Turning it into an agent

In [24]:
def search_tool(query: str) -> str:
    """Search the course lesson knowledge base for specific information chunks."""
    results = index_chunks.search(query=query, num_results=2)
    return "\n\n".join([f"File: {r['filename']}\nContent: {r['content']}" for r in results])

agent_instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "course_search_tool tool. Make multiple searches with different keywords before answering."
)

agent_query = "How does the agentic loop work, and how is it different from plain RAG?"

agent_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": agent_instructions},
        {"role": "user", "content": agent_query}
    ])

message = agent_response.choices[0].message

tool_calls = getattr(message, "tool_calls", None)

print("Tools called:", len(tool_calls) if tool_calls else 0)

Tools called: 0
